## check category distribution

In [1]:
import pandas as pd

chunksize = 1_000_000
counts = {}

for chunk in pd.read_csv(
        "exo_CTL_08.01.csv",
        header=None,
        usecols=[2],
        chunksize=chunksize):

    vc = chunk[2].value_counts()

    for k, v in vc.items():
        counts[k] = counts.get(k, 0) + v

result = (
    pd.DataFrame(
        counts.items(),
        columns=["Category", "Count"]
    )
    .sort_values("Count", ascending=False)
)

print(result)
result.to_csv("category_counts.csv", index=False)

          Category    Count
0  planetcandidate  5545136
1    cooldwarfs_v8  3940291
2  hotsubdwarfs_v8     2855


# creating sample dataset from the large dataset

In [8]:
import pandas as pd

chunksize = 500_000
sample_parts = []

for chunk in pd.read_csv(
        "exo_CTL_08.01.csv",
        header=None,
        chunksize=chunksize):

    sample_parts.append(
        chunk.groupby(2, group_keys=False)
             .sample(frac=0.01, random_state=42)
    )

sample = pd.concat(sample_parts)

sample.to_csv(
    "training_sample.csv",
    index=False,
    header=False
)

print(len(sample))

94883


# Random Forest

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv(
    "training_sample.csv",
    header=None
)

X = df[[1]]        # probability column
y = df[2]          # category

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.58      0.25      0.35      7881
           1       0.00      0.00      0.00         6
           2       0.62      0.87      0.72     11090

    accuracy                           0.61     18977
   macro avg       0.40      0.37      0.36     18977
weighted avg       0.60      0.61      0.57     18977



c:\Users\absol\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\absol\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\absol\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

# xgboost

In [14]:
from xgboost import XGBClassifier
import joblib

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss"
)

xgb.fit(X_train, y_train)

pred = xgb.predict(X_test)

print(classification_report(y_test, pred))
joblib.dump(xgb, "star_classifier.pkl")
joblib.dump(le, "label_encoder.pkl")

              precision    recall  f1-score   support

           0       0.69      0.60      0.64      7881
           1       0.00      0.00      0.00         6
           2       0.74      0.81      0.77     11090

    accuracy                           0.72     18977
   macro avg       0.48      0.47      0.47     18977
weighted avg       0.72      0.72      0.72     18977



c:\Users\absol\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\absol\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\absol\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

['label_encoder.pkl']

# for RAW 9 million Data set

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    random_state=42
)

# Model Load

In [15]:
import joblib

model = joblib.load("star_classifier.pkl")
le = joblib.load("label_encoder.pkl")

In [23]:
print(model.feature_names_in_)

['1']


In [24]:
import pandas as pd

unknown = pd.DataFrame(
    {1: [9.53806661475452e-05]}
)

pred = model.predict(unknown)


category = le.inverse_transform(pred)

print(category[0])

cooldwarfs_v8
